# Theoretical visualization: what the two stages fit

Builds the two schematic paper figures `theory_two_stage_ellipses.pdf` (two-stage fit of a
bivariate marginal) and `theory_flow_panels.pdf` (the method flowchart); no data and no
training. The predictive covariance is
$\Sigma_P = D\,C\,D$ with $D=\operatorname{diag}(\tau)$ the marginal scale and
$C = F^{*}\operatorname{diag}(s)F$, $\operatorname{mean}(s)=1$, a circulant
correlation core. Stage 1 fits $\tau$ on the marginals; stage 2 fits $s$ on the
standardized target. The population stage-2 minimizer of the log score is
$s^\star_k=(FRF^{*})_{kk}$, i.e. $C^\star=\Pi_{\mathrm{circ}}(R)$ — the
Frobenius projection of the true correlation onto the circulant matrices, which
in physical space averages $R$ over all pairs at the same lag.

**Why the ambient truth is $d=3$ and not $d=2$.** Every $2\times2$ correlation
matrix is already circulant, so at $d=2$ the projection is the identity and
stage 2 would reproduce $\rho$ exactly — nothing to see. $d=3$ is the smallest
truth on which the approximation exists: the core must give all three pairs one
common correlation, so it fits their average. The figure itself is purely
two-dimensional: it shows coordinates $1,2$, whose exact bivariate marginal is
$\mathcal N(0,\Sigma_{\{1,2\}})$, parameterized by $\sigma_1,\sigma_2,\rho$.

In [ ]:
import numpy as np
import plot_config as pc
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

cfg = pc.apply_style()

# Stage colours are figure-local (per plot_config's convention); the truth and the
# stage-2 arm take their semantic roles from the shared palette.
C_TRUE, C_STAGE1 = pc.COLORS["truth"], "#d95f02"
C_CORE, C_STAGE2 = "#7570b3", pc.COLORS["kernel_score"]
C_BOX = pc.COLORS["guide"]

TAU = np.array([1.0, 1.7, 0.8])            # marginal scales; 1,2 are the plotted pair
R_TRUE = np.array([[1.00, 0.70, 0.40],     # true correlation: not circulant
                   [0.70, 1.00, 0.55],
                   [0.40, 0.55, 1.00]])
LEVEL = 2.0                                # Mahalanobis radius of every contour

d = len(TAU)
D = np.diag(TAU)
F = np.fft.fft(np.eye(d), norm="ortho")

s_star = np.real(np.diag(F @ R_TRUE @ F.conj().T))          # stage-2 minimizer
C_STAR = np.real(F.conj().T @ np.diag(s_star) @ F)          # = Pi_circ(R)
SIGMA_Q = D @ R_TRUE @ D                                    # truth
SIGMA_1 = D @ np.eye(d) @ D                                 # stage 1: marginals only
SIGMA_2 = D @ C_STAR @ D                                    # stage 2: predictive covariance

assert np.all(np.linalg.eigvalsh(R_TRUE) > 0) and np.all(s_star > 0)
assert np.isclose(s_star.mean(), 1.0)                       # unit-mean gauge, for free
assert np.allclose(np.diag(C_STAR), 1.0)                    # marginals survive stage 2
assert np.allclose(np.diag(SIGMA_2), np.diag(SIGMA_Q))
assert np.isclose(C_STAR[0, 1], R_TRUE[np.triu_indices(d, 1)].mean())   # lag average
assert abs(np.sum((R_TRUE - C_STAR) * C_STAR)) < 1e-12      # orthogonal residual

RHO_Q, RHO_P = R_TRUE[0, 1], C_STAR[0, 1]
print(f"rho: {RHO_Q:.2f} (truth) -> {RHO_P:.2f} (stage 2),  stage 1: 0")

In [ ]:
def ellipse(cov, level=LEVEL, n=400):
    """Points on the contour {x : x^T cov^{-1} x = level^2}."""
    t = np.linspace(0, 2 * np.pi, n)
    return (level * np.linalg.cholesky(cov) @ np.stack([np.cos(t), np.sin(t)])).T


def density(cov, lim, n=240):
    """Gaussian density on a square grid, scaled to a maximum of 1."""
    g = np.linspace(-lim, lim, n)
    X, Y = np.meshgrid(g, g)
    P = np.linalg.inv(cov)
    return X, Y, np.exp(-0.5 * (P[0, 0] * X**2 + 2 * P[0, 1] * X * Y + P[1, 1] * Y**2))


# Fill levels start above zero, so the tails stay unfilled and the panel keeps a
# white background instead of a grey wash.
FILL = np.linspace(0.06, 1.0, 9)


pair = np.ix_([0, 1], [0, 1])
fig, axes = pc.subplots(1, 2, aspect=0.50, columns="double")

# (a) physical space -------------------------------------------------------------
ax, lim = axes[0], 1.12 * LEVEL * TAU[1]
X, Y, Z = density(SIGMA_Q[pair], lim)
ax.contourf(X, Y, Z, levels=FILL, cmap=pc.grey_ramp(0.97, 0.68), zorder=0)
ax.add_patch(Rectangle((-LEVEL * TAU[0], -LEVEL * TAU[1]),
                       2 * LEVEL * TAU[0], 2 * LEVEL * TAU[1], fill=False,
                       ec=C_BOX, ls=(0, (4, 3)), lw=cfg.thin_line_width, zorder=1))
for cov, col, ls in [(SIGMA_Q[pair], C_TRUE, "-"),
                     (SIGMA_1[pair], C_STAGE1, (0, (5, 2))),
                     (C_STAR[pair], C_CORE, (0, (1, 1.6))),
                     (SIGMA_2[pair], C_STAGE2, "-")]:
    ax.plot(*ellipse(cov).T, color=col, ls=ls, lw=cfg.line_width, zorder=3)
ax.set_xlabel("$y_1$"); ax.set_ylabel("$y_2$")
ax.set_title("(a) physical space")

# (b) standardized space -------------------------------------------------------------
ax, lim_w = axes[1], 1.28 * LEVEL
X, Y, Z = density(R_TRUE[pair], lim_w)
ax.contourf(X, Y, Z, levels=FILL, cmap=pc.grey_ramp(0.97, 0.68), zorder=0)
ax.add_patch(Rectangle((-LEVEL, -LEVEL), 2 * LEVEL, 2 * LEVEL, fill=False,
                       ec=C_BOX, ls=(0, (4, 3)), lw=cfg.thin_line_width, zorder=1))
for cov, col, ls in [(R_TRUE[pair], C_TRUE, "-"),
                     (np.eye(2), C_STAGE1, (0, (5, 2))),
                     (C_STAR[pair], C_STAGE2, "-")]:
    ax.plot(*ellipse(cov).T, color=col, ls=ls, lw=cfg.line_width, zorder=3)
ax.set_xlabel(r"$z_1 = y_1/\tau_1$"); ax.set_ylabel(r"$z_2 = y_2/\tau_2$")
ax.set_title("(b) whitened space")

for ax, l in zip(axes, [lim, lim_w]):
    ax.set_aspect("equal"); ax.set_xlim(-l, l); ax.set_ylim(-l, l)

fig.legend(
    handles=[
        Line2D([], [], color=C_TRUE, lw=cfg.line_width,
               label=rf"truth ($\rho={RHO_Q:.2f}$)"),
        Line2D([], [], color=C_STAGE1, lw=cfg.line_width, ls=(0, (5, 2)),
               label=r"stage 1 ($\rho=0$)"),
        Line2D([], [], color=C_CORE, lw=cfg.line_width, ls=(0, (1, 1.6)),
               label=r"core $C^\star$"),
        Line2D([], [], color=C_STAGE2, lw=cfg.line_width,
               label=rf"stage 2 ($\rho={RHO_P:.2f}$)"),
        Line2D([], [], color=C_BOX, lw=cfg.thin_line_width, ls=(0, (4, 3)),
               label=rf"${LEVEL:g}\tau$ box"),
    ],
    ncol=5, loc="outside lower center", columnspacing=1.1, handlelength=1.9,
)

#pc.save(fig, "theory_two_stage_ellipses.pdf")

**Draft caption.** *Two-stage fit of a bivariate marginal of a three-variable
Gaussian truth ($\sigma_1=1.0$, $\sigma_2=1.7$, $\rho=0.70$; the remaining true
correlations are $0.55$ and $0.40$), at Mahalanobis radius $2$.* **(a)** Stage 1
scores the marginals only, so it recovers $\tau$ exactly and leaves the
dependence at zero. Stage 2 then fits the circulant correlation core $C^\star$
(dotted, on its own unit scale) and returns the predictive covariance $\Sigma_2 = DC^\star\!D$. All three
ellipses touch the marginal box: the unit-mean gauge makes
$\operatorname{diag}C^\star=\mathbf 1$, so stage 2 returns the stage-1 marginals
untouched and the whole approximation error is the tilt. **(b)** The same
objects for the standardized target, where stage 2 works: $C^\star=\Pi_{\mathrm{circ}}(R)$
is the orthogonal projection of the true correlation onto the circulant cores,
here the average of the three true pair correlations, $\rho=0.70\to0.55$.

---
## Method flowchart

Shared helpers for the flowchart figures below. Every node is
a *floating* contour on one common scale `SC`, so
the changes in size and shape along the chain **are** the transforms rather than
a per-node normalization, and a thin grey copy of the truth sits behind each fit
so the residual error is visible at every step.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.path import Path
# An engine *instance* survives savefig's layout restore; layout="none" comes back
# as constrained layout under pc.apply_style.
from matplotlib.layout_engine import PlaceHolderLayoutEngine

SC = 0.22            # data units -> diagram units, shared by every floating ellipse
C_ARROW = "0.35"
EYE = np.eye(2)
pair2 = np.ix_([0, 1], [0, 1])

# The spectral domain: stage 2 keeps the diagonal of F R F* and discards the rest.
FRF = F @ R_TRUE @ F.conj().T
S_STAR = np.real(np.diag(FRF))


def floating(ax, center, cov, color, ls="-", lw=None, sc=SC, z=3):
    """A 2-sigma contour of `cov`, free-standing at `center`; no axes, no ticks."""
    ax.plot(*(np.asarray(center, float) + sc * ellipse(cov)).T,
            color=color, ls=ls, lw=lw or cfg.line_width, zorder=z)


def half(cov, sc=SC):
    """(half-width, half-height) of the drawn contour, for placing labels/arrows."""
    e = sc * ellipse(cov)
    return e[:, 0].max(), e[:, 1].max()


def label(ax, xy, text, color="black", size=None, **kw):
    kw.setdefault("ha", "center")
    kw.setdefault("va", "center")
    ax.text(*xy, text, color=color, zorder=6, size=size or cfg.annotation_size, **kw)


def arrow(ax, tail, head, color=C_ARROW, lw=None, rad=0.0):
    ax.add_patch(FancyArrowPatch(tail, head, arrowstyle="-|>", mutation_scale=9,
                                 color=color, lw=lw or cfg.thin_line_width,
                                 shrinkA=1, shrinkB=1, zorder=2,
                                 connectionstyle=f"arc3,rad={rad}"))


def elbow(ax, verts, color=C_ARROW, lw=None):
    """Right-angled routed arrow through `verts` (a list of corner points)."""
    ax.add_patch(FancyArrowPatch(path=Path(verts), arrowstyle="-|>",
                                 mutation_scale=9, color=color,
                                 lw=lw or cfg.thin_line_width, zorder=2))


def spectrum(ax, center, s, w=0.26, h=0.5):
    """Bar glyph of the fitted spectrum, with the unit-mean gauge as a dashed line."""
    x0, y0 = center
    xs = x0 + w * (np.arange(len(s)) - (len(s) - 1) / 2)
    ax.bar(xs, h * np.asarray(s) / max(s), bottom=y0, width=0.68 * w,
           color=C_CORE, alpha=0.8, zorder=3)
    ax.plot([xs[0] - 0.55 * w, xs[-1] + 0.55 * w], [y0 + h / max(s)] * 2,
            color=C_BOX, ls=(0, (3, 2)), lw=cfg.thin_line_width, zorder=4)


def node(ax, center, layers, title, title_dy=0.16):
    """Draw the stacked contours of one node and its title, placed above it."""
    top = 0.0
    for cov, color, ls, lw in layers:
        floating(ax, center, cov, color, ls=ls, lw=lw)
        top = max(top, half(cov)[1])
    label(ax, (center[0], center[1] + top + title_dy), title, va="bottom")
    return top


# Tinted background band per stage, plus the predictive: fill colour and the colour
# of its label.
BAND = {"Stage 1": ("#fbeadd", C_STAGE1), "Stage 2": ("#e9e6f4", C_CORE),
        "Predictive": ("#dcefe7", C_STAGE2)}
TICK = dict(labelsize=5.8, length=2, pad=1.5)

# One colour key for every flowchart below; each fit is named by its panel.
KEY = [Line2D([], [], color=C_TRUE, lw=cfg.line_width, label="Truth"),
       Line2D([], [], color=C_STAGE1, lw=cfg.line_width, ls=(0, (5, 2)),
              label=r"Marginals $D^2$"),
       Line2D([], [], color=C_CORE, lw=cfg.line_width, ls=(0, (1, 1.6)),
              label=r"Core $R^\star$"),
       Line2D([], [], color=C_STAGE2, lw=cfg.line_width, ls=(0, (5, 2)),
              label=r"Predictive $\Sigma_P$")]

### The method flowchart

One framed panel per node, in a single row, 6.75 x 1.63 in. The vertical layout
is written in inches (see `fy`), so `H_IN` is the single knob for the height.

* Each ellipse panel is filled with the **density of the truth** and carries the
  truth's contour, so every panel is a fit-against-truth comparison rather than
  an outline. The truth is solid, every fit is dashed (the core, which is never
  used as the predictive covariance as-is, stays dotted).
* Three tinted bands, **Stage 1**, **Stage 2** and **Predictive**, so the arrows
  only have to name the operators, **$F$ and $F^{*}$** included. Each stage band
  takes in the arrow that feeds it and runs on over the next one; one clearance
  (`BOX_GAP`) separates every band from its neighbour, and no band edge may cut an
  arrow label (asserted).
* **Two text rows** above every panel, on fixed baselines: row 1 in bold names the
  stage (or *Target* / *Predictive* outside the stages), row 2 is the panel's title
  or formula.
* Panels carry a grid but **no tick numbers**, so the axis ranges belong in the
  caption: the physical panels span $\pm4$ and the standardized ones $\pm2.6$. Worth
  stating there, because without the numbers the standardized panels look like the
  same scale as the physical ones when they are not — the only tick numbers left
  are the mode index under the spectral panel and the $1$ marking the unit-mean
  gauge.
* With nothing sticking into the gaps, every gap is equal and every arrow uses
  **one clearance at both ends**, so all five arrows are identical in length and
  spacing; labels are centred on their arrow.
* The $D$ rail underneath carries the point that
  stage 1's $D$ is frozen and reused in the predictive covariance: it feeds the $D(\cdot)D$
  arrow from below.

Titles are coloured to match their contour, so the legend only carries the line
styles. `draw_flowchart` takes a `card` colour, which is what the grey version
below uses; everything else about the two figures is identical.

In [ ]:
def panel_x(fig, rect, lim, title, color, ticks=(-3, 0, 3), labels=True, truth=None):
    """A framed node with the truth's density behind it and a coloured title."""
    ax = fig.add_axes(rect)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.set_facecolor("white")                # stays white on the grey card
    ax.set_xticks(ticks); ax.set_yticks(ticks)
    ax.tick_params(**TICK, labelbottom=labels, labelleft=labels)
    if truth is not None:
        X, Y, Z = density(truth, lim)
        ax.contourf(X, Y, Z, levels=FILL, cmap=pc.grey_ramp(0.99, 0.74), zorder=0)
    # Grid on top of the density fill, or the fill would bury it.
    ax.set_axisbelow(False)
    ax.grid(True, lw=cfg.axes_line_width, alpha=cfg.grid_alpha)
    ax.set_title(title, size=cfg.annotation_size, pad=2.5, color=color)
    return ax


# --- vertical layout, written in inches from the bottom of the canvas ----------
# Everything below the panels (legend, rail, tick labels) is text at a fixed point
# size, so its height in inches is fixed while its figure *fraction* moves with
# H_IN. Stating the layout in inches keeps it readable and makes H_IN the single
# knob for the figure's height.
W_IN = pc.width_of("double")                # 6.75 in: the paper text width
H_IN = 1.63                                 # total figure height
ASPECT = H_IN / W_IN


def fy(y_in):
    """Inches above the canvas bottom -> figure fraction."""
    return y_in / H_IN


def fx(x_in):
    """Inches from the canvas left -> figure fraction."""
    return x_in / W_IN


WX = 0.095                                  # panel width, figure fraction
HX = WX * W_IN / H_IN                       # ... and the height that squares it
GAP = 0.076                                 # equal everywhere: 0.51 in
CLEAR = 0.010                               # one clearance at both ends of every arrow

# Two text rows above every panel, on fixed baselines: row 1 names the stage (or
# the node, outside the stages) in bold, row 2 is the panel title. The panel row
# sits low, in what would otherwise be slack under the legend, which is what buys
# the gap between the two rows.
Y_LEGEND = fy(0.045)                        # legend anchor, bottom edge
RAILY = fy(0.353)                           # the D rail
ROWY = fy(0.613)                            # panel bottoms
ROW1_Y = fy(1.490)                          # row-1 baseline
BAND_Y = (fy(0.478), fy(1.580))             # tinted stage bands
ARROWS = [r"Fit $\mu,\tau$", r"$D^{-1}$", r"$F$", r"$F^{*}$", r"$D(\cdot)D$"]
CARD = (0.010, fy(0.027), 0.990, fy(1.608))  # x0, y0, x1, y1 of the rounded box

# The green band overhangs the last panel, so the row is centred on the card with
# an overhang budgeted on the right -- otherwise the band crowds the card's edge.
ROW_W = 6 * WX + 5 * GAP
OVERHANG = 0.009
MARGIN = (CARD[2] - CARD[0] - ROW_W - OVERHANG) / 2
XS = [CARD[0] + MARGIN + k * (WX + GAP) for k in range(6)]   # Q, D2, R, spectral, R*, P

# Each band takes in the arrow that feeds its stage and runs on over the next one.
# One clearance, BOX_GAP, separates every band from its neighbour, panel or band:
# Stage 1 sits BOX_GAP from both of its panels, so it swallows all of D^{-1};
# Stage 2 stops just past the D(.)D label (no band edge may cut text), and the
# green band follows BOX_GAP later, symmetric about the predictive panel.
BOX_GAP = fx(0.050)
LABEL_HALF_DD = fx(0.153)                   # measured half-width of "D(.)D"
_mid = lambda k: (XS[k] + WX + XS[k + 1]) / 2          # centre of arrow k
BAND_X = {"Stage 1": (XS[0] + WX + BOX_GAP, XS[2] - BOX_GAP),
          "Stage 2": (XS[2] + WX + BOX_GAP, _mid(4) + LABEL_HALF_DD + fx(0.016))}
BAND_X["Predictive"] = (BAND_X["Stage 2"][1] + BOX_GAP,
                        2 * XS[5] + WX - BAND_X["Stage 2"][1] - BOX_GAP)

def draw_flowchart(card=None):
    """Build the flowchart.

    `card` draws a rounded box of that colour behind everything, inset from the
    canvas edges; `None` leaves the figure on plain white.
    """
    paper = card or "white"
    fig = plt.figure(figsize=pc.figsize(ASPECT, "double"),
                     layout=PlaceHolderLayoutEngine(True, True))
    bg = fig.add_axes([0, 0, 1, 1]); bg.axis("off")
    bg.set_xlim(0, 1); bg.set_ylim(0, 1); bg.patch.set_alpha(0)

    if card is not None:
        # bg spans the canvas with unit limits, so one figure-fraction step is
        # much longer in x than in y; mutation_aspect un-squashes the corners so
        # the rounding reads as circular on the page rather than stretched.
        w_in, h_in = pc.figsize(ASPECT, "double")
        bg.add_patch(FancyBboxPatch((CARD[0], CARD[1]),
                                    CARD[2] - CARD[0], CARD[3] - CARD[1],
                                    boxstyle="round,pad=0,rounding_size=0.012",
                                    mutation_aspect=w_in / h_in, fc=card,
                                    ec="#e2e2de", lw=cfg.axes_line_width,
                                    zorder=-5))

    for name, (x0, x1) in BAND_X.items():
        bg.add_patch(Rectangle((x0, BAND_Y[0]), x1 - x0, BAND_Y[1] - BAND_Y[0],
                               fc=BAND[name][0], ec="none", zorder=0))

    # Row 1: centred on the panels it names, not on the band, which is lopsided.
    row1 = []
    for text, xc, color in [(r"\textbf{Target}", XS[0] + WX / 2, C_TRUE),
                            (r"\textbf{Stage 1}", XS[1] + WX / 2, C_STAGE1),
                            (r"\textbf{Stage 2}", (XS[3] + XS[4] + WX) / 2, C_CORE),
                            (r"\textbf{Predictive}", XS[5] + WX / 2, C_STAGE2)]:
        row1.append(bg.text(xc, ROW1_Y, text, ha="center", va="baseline",
                            color=color, size=cfg.annotation_size, zorder=1))

    axQ = panel_x(fig, [XS[0], ROWY, WX, HX], 4.0, r"$\Sigma_Q$", C_TRUE,
                  labels=False, truth=SIGMA_Q[pair2])
    axD = panel_x(fig, [XS[1], ROWY, WX, HX], 4.0, r"Marginals: $D^2$", C_STAGE1,
                  labels=False, truth=SIGMA_Q[pair2])
    axW = panel_x(fig, [XS[2], ROWY, WX, HX], 2.6, r"$D^{-1}\Sigma_Q D^{-1}$", C_TRUE,
                  (-2, 0, 2), labels=False, truth=R_TRUE[pair2])
    axS = fig.add_axes([XS[3], ROWY, WX, HX])
    axC = panel_x(fig, [XS[4], ROWY, WX, HX], 2.6, r"Core: $R^\star$", C_CORE,
                  (-2, 0, 2), labels=False, truth=R_TRUE[pair2])
    axP = panel_x(fig, [XS[5], ROWY, WX, HX], 4.0, r"$\Sigma_P$",
                  C_STAGE2, labels=False, truth=SIGMA_Q[pair2])

    for ax, layers in [(axQ, [(SIGMA_Q[pair2], C_TRUE, "-")]),
                       (axD, [(SIGMA_Q[pair2], C_TRUE, "-"),
                              (SIGMA_1[pair2], C_STAGE1, (0, (5, 2)))]),
                       (axW, [(R_TRUE[pair2], C_TRUE, "-"),
                              (EYE, C_STAGE1, (0, (5, 2)))]),
                       (axC, [(C_STAR[pair2], C_CORE, (0, (1, 1.6)))]),
                       (axP, [(SIGMA_Q[pair2], C_TRUE, "-"),
                              (SIGMA_2[pair2], C_STAGE2, (0, (5, 2)))])]:
        for cov, color, ls in layers:
            ax.plot(*ellipse(cov).T, color=color, ls=ls, lw=cfg.line_width, zorder=3)

    # Spectral node: the mode index stays, the y numbers go (they were the only
    # thing left reaching into an arrow gap); the gauge line is labelled instead.
    axS.set_facecolor("white")
    axS.bar(np.arange(d), S_STAR, width=0.55, color=C_CORE, alpha=0.8, zorder=3)
    axS.axhline(1.0, color=C_BOX, ls=(0, (3, 2)), lw=cfg.thin_line_width, zorder=4)
    axS.text(2.45, 1.06, "$1$", ha="right", va="bottom", size=5.8, color="0.35",
             zorder=5)
    axS.set_xticks(np.arange(d)); axS.set_yticks([0, 1, 2])
    axS.set_xlim(-0.6, d - 0.4)
    axS.tick_params(**TICK, labelleft=False)
    axS.grid(True, lw=cfg.axes_line_width, alpha=cfg.grid_alpha)
    axS.set_title(r"Spectral: $s^\star_k$", size=cfg.annotation_size, pad=2.5,
                  color=C_CORE)

    def fig_arrow(a, b):
        bg.add_patch(FancyArrowPatch(a, b, arrowstyle="-|>", mutation_scale=8,
                                     color=C_ARROW, lw=cfg.thin_line_width,
                                     shrinkA=0, shrinkB=0, zorder=2))

    ym = ROWY + HX / 2
    labels = []
    for k, lab in enumerate(ARROWS):
        xa, xb = XS[k] + WX + CLEAR, XS[k + 1] - CLEAR
        fig_arrow((xa, ym), (xb, ym))
        labels.append(bg.text((xa + xb) / 2, ym + 0.020, lab, ha="center",
                              va="bottom", size=5.8, color=C_ARROW, zorder=3))

    # The rail leaves the orange band and feeds D into the D(.)D arrow from below.
    bg.add_patch(FancyArrowPatch(path=Path([(XS[1] + WX / 2, BAND_Y[0] - 0.004),
                                            (XS[1] + WX / 2, RAILY),
                                            (_mid(4), RAILY),
                                            (_mid(4), ym - fy(0.012))]),
                                 arrowstyle="-|>", mutation_scale=8, color=C_ARROW,
                                 lw=cfg.thin_line_width, zorder=2))
    # Sits on the rail with the card colour behind it, so the line reads as broken
    # rather than struck through the text.
    bg.text(0.58, RAILY, r"$D$ from Stage 1, frozen", ha="center", va="center",
            size=5.8, color=C_ARROW, zorder=3,
            bbox=dict(fc=paper, ec="none", pad=1.2))

    leg = fig.legend(handles=KEY, ncol=4, loc="lower center",
                     bbox_to_anchor=(0.5, Y_LEGEND), handlelength=1.6,
                     columnspacing=1.2, borderpad=0.35, fontsize=6.5)

    # The layout is hand-placed, so assert what the eye is meant to check.
    fig.canvas.draw()
    r, inv = fig.canvas.get_renderer(), fig.transFigure.inverted()
    ext = lambda a: a.get_window_extent(r).transformed(inv)
    for name, ax in [("Stage 1", axD), ("Stage 2", axS), ("Stage 2", axC),
                     ("Predictive", axP)]:
        t = ext(ax.title)
        assert BAND_Y[0] < t.y0 and t.y1 < BAND_Y[1], (
            f"{name} title escapes its band: {t.y0:.3f}-{t.y1:.3f} vs {BAND_Y}")
        ticks = [ext(l).y0 for l in ax.get_xticklabels() if l.get_text()]
        assert not ticks or min(ticks) > BAND_Y[0], (
            f"{name} tick labels dip below its band: {min(ticks):.3f}")
    # Row 1 clears row 2 everywhere, and the stage headers stay inside their band.
    row2_top = max(ext(ax.title).y1 for ax in (axQ, axD, axW, axS, axC, axP))
    for t in row1:
        e = ext(t)
        assert e.y0 > row2_top, f"row 1 {t.get_text()!r} touches row 2"
        assert e.y1 < BAND_Y[1], f"row 1 {t.get_text()!r} escapes the band top"
    for t, name in zip(row1[1:], BAND_X):
        e, (x0, x1) = ext(t), BAND_X[name]
        assert x0 < e.x0 and e.x1 < x1, f"{name} header leaves its band"
    # Each stage band takes in the tail of the arrow that feeds it.
    for name, k in [("Stage 1", 0), ("Stage 2", 2)]:
        assert BAND_X[name][0] < XS[k] + WX + CLEAR, f"{name} misses its arrow"
    for k, t in enumerate(labels):
        e = ext(t)
        assert XS[k] + WX < e.x0 and e.x1 < XS[k + 1], (
            f"arrow label {k} overlaps a panel: {e.x0:.3f}-{e.x1:.3f}")
        # A band edge in the gap must not cut through this gap's label.
        for x0, x1 in BAND_X.values():
            for edge in (x0, x1):
                assert not (e.x0 - 0.002 < edge < e.x1 + 0.002), (
                    f"a band edge cuts arrow label {k}: {edge:.4f} in "
                    f"{e.x0:.4f}-{e.x1:.4f}")
    boxes = [ext(leg)] + [Rectangle((x0, BAND_Y[0]), x1 - x0,
                                    BAND_Y[1] - BAND_Y[0]).get_bbox()
                          for x0, x1 in BAND_X.values()]
    for ax in (axQ, axD, axW, axS, axC, axP):
        boxes += [ax.get_position(), ext(ax.title)]
    for e in boxes:
        assert CARD[0] <= e.x0 and e.x1 <= CARD[2], (
            f"content escapes the card in x: {e.x0:.4f}-{e.x1:.4f} vs "
            f"{CARD[0]}-{CARD[2]}")
        assert CARD[1] <= e.y0 and e.y1 <= CARD[3], (
            f"content escapes the card in y: {e.y0:.4f}-{e.y1:.4f} vs "
            f"{CARD[1]}-{CARD[3]}")
    return fig


fig = draw_flowchart()
pc.save(fig, "theory_flow_panels.pdf")

### Same figure on a grey card

Identical geometry, with a rounded grey box inset behind everything and the
panels held at white so they read as cards on it. Whether it helps is a
judgement call: it groups the row into one object and makes the panels pop, but
it costs the two stage bands some contrast, since they were the only fill in the
figure and now sit on tint rather than on white.

In [ ]:
fig = draw_flowchart(card="#f2f2f0")
pc.save(fig, "theory_flow_panels.pdf")